# 30. 제목 경계 청킹과 parent-child 비교 — 검색 5:5 고정

**목적:** 작은 청크를 그대로 반환하는 방식과, 작은 자식으로 검색한 뒤 큰 부모 문맥을 반환하는 방식을 비교합니다.
검색 조건은 사용자와 합의한 E5:Kiwi BM25 = **5:5**, 각 arm 후보 20개, RRF k=60으로 고정합니다.

| 후보 | 검색·임베딩 단위 | 검색 후 반환 문맥 |
|---|---|---|
| A `title_boundary` | 제목 안에서 400-token SentenceSplitter, overlap 48 | 검색된 작은 청크 그대로 |
| B `parent_child` | 제목 안에 1,200-token 부모를 만들고, 그 안의 400-token 자식으로 검색 | 상위 자식의 부모를 가져오고 동일 부모는 한 번만 반환 |

두 후보 모두 제목 경계를 지킵니다. B는 LlamaIndex **HierarchicalNodeParser**를 사용하며 부모·자식 ID 연결을 실제로 생성합니다.
B는 부모 분할 뒤 자식을 나누므로 A와 자식 경계도 달라질 수 있습니다. 따라서 **계층 분할 + 부모 확장의 구성 비교**이며, 부모 확장만의 독립 효과는 아닙니다.
부모 1,200/자식 400은 이번 첫 비교 설정이지, 모든 크기를 최적화한 최종값이 아닙니다. AutoMergingRetriever의 임계값 기반 병합은 사용하지 않습니다.

**평가 범위:** 29번과 같은 5개 회사·2024 사업보고서와 기존 질문 25개, 기존 정답 근거를 묶은 복합 질문 5개.
총 30개 개발 질문이며 독립 holdout이 아닙니다. 단일/복합 결과를 분리합니다. 복합 질문의 근거 목록은 검색 전에 고정합니다.

**공정성:** 상위 **자식/작은 청크 5개**까지 검색하고 B는 그 부모를 중복 제거합니다. 부모 5개가 나올 때까지 더 검색하지 않습니다.
원래 반환 문맥과 **공통 2,400 E5-token 예산**의 반환 문맥을 따로 평가합니다. 예산은 비교용이며 최종 운영 제한이 아닙니다.
앞에서부터 문맥 단위를 통째로 넣고, 다음 단위가 예산을 넘으면 멈춥니다. 문장 중간을 잘라 정답을 훼손하지 않습니다.

**판단 지표:** 검색 청크 Hit@5·MRR@5, 반환 문맥의 Hit@5·고정 근거 Recall@5, 예산 적용 후 같은 지표, 전달 토큰량과 지연시간.
Recall은 후보별 청크 개수가 아닌 **동일한 정답 인용문 ID 목록**을 분모로 써 비교합니다. 의미상 모든 관련 근거를 판정한 완전한 relevance는 아닙니다.
단일 질문은 정답 인용문이 1개라 Hit와 Recall이 같고, 복합 질문은 2개 중 몇 개를 찾았는지 구분됩니다.

**제외:** 기업·연도 식별, 구조화 SQL 검색, reranker, LLM 답변·할루시네이션 평가는 없습니다. 부모는 E5에 임베딩하지 않습니다.
`skn25`에서 위에서 아래로 실행·저장하세요. Codex는 저장된 결과를 보고 해석하고 그때 HTML 보고서를 작성합니다.
OpenDART만 호출하며 OpenAI/LLM 과금과 Qwen GPU 설정 변경은 없습니다. 기존 notebook 전체·출력은 실행하지 않습니다.

## 1. 환경·검증된 함수와 데이터 계약 재사용

28번에서 원문 추출·E5·Kiwi·BM25 함수 5개, 29번에서 RRF 함수·고정 데이터·공식 보고서 선택 코드를 재사용합니다.
AST 해시를 실행 전에 검사합니다. 함수·데이터 계약이 바뀌면 중단하며, 기존 notebook의 출력 저장만 바뀐 것은 허용합니다.
28·29번을 먼저 실행할 필요는 없습니다. 이것은 실험용 notebook 의존성이며 최종 운영 코드의 방식이 아닙니다.

In [ ]:
from io import BytesIO
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import ast, hashlib, importlib.metadata, json, os, re, sqlite3, sys, zipfile
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
from IPython.display import display
from kiwipiepy import Kiwi
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser, get_leaf_nodes
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F

if Path(sys.prefix).name != 'skn25':
    raise RuntimeError('conda skn25 커널에서만 실행하세요.')
if importlib.metadata.version('llama-index-core') != '0.14.25':
    raise RuntimeError('검증한 llama-index-core 0.14.25와 다릅니다. 계층 파서 재검토가 필요합니다.')
PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'src/dart/client.py').is_file())
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env의 유효한 OPENDART_API_KEY가 필요합니다.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from dart.client import _get_bytes, CORP_CODE_URL

nb28 = json.loads((PROJECT_ROOT / 'notebooks/v1/28_hybrid_retrieval_paired_comparison.ipynb').read_text())
nb29 = json.loads((PROJECT_ROOT / 'notebooks/v1/29_rrf_weight_comparison.ipynb').read_text())
core_names = {'download_xml', 'extract_b_text', 'morph_terms', 'encode_texts', 'bm25_order'}
helpers = [n for c in nb28['cells'] if c['cell_type'] == 'code'
           for n in ast.parse(''.join(c['source'])).body
           if isinstance(n, ast.FunctionDef) and n.name in core_names]
helpers += [n for c in nb29['cells'] if c['cell_type'] == 'code'
            for n in ast.parse(''.join(c['source'])).body
            if isinstance(n, ast.FunctionDef) and n.name == 'weighted_rrf']
data_nodes = [n for c in nb29['cells'] if c['cell_type'] == 'code'
              for n in ast.parse(''.join(c['source'])).body
              if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id in {'REPORTS','CASES'} for t in n.targets)]
source_gate = ast.parse(''.join(next(c for c in nb29['cells'] if c.get('id') == 'v29-06')['source']))
reused_ast = ast.Module(body=helpers + data_nodes + source_gate.body, type_ignores=[])
reuse_sha256 = hashlib.sha256(ast.dump(reused_ast).encode()).hexdigest()
if len(helpers) != 6 or len(data_nodes) != 2 or reuse_sha256 != '0c51db7110a13828d60f8a92999c33ef19661bc0dc5d7fce16c68862082c3eab':
    raise RuntimeError('재사용 함수·데이터 계약이 검토된 버전과 다릅니다. 재검토 전 실행하지 않습니다.')
MODEL_ID, MODEL_REVISION = 'intfloat/multilingual-e5-small', '5697a65b0a002a92fe8c4fc9d495303ffff9c7d2'
CANDIDATE_DEPTH, RRF_K, EVAL_K = 20, 60, 5
VECTOR_WEIGHT, CHILD_SIZE, PARENT_SIZE, OVERLAP, CONTEXT_BUDGET = 0.5, 400, 1200, 48, 2400
METHODS = ('title_boundary', 'parent_child')
compact = lambda text: re.sub(r'[\s|]+', '', text)
exec(compile(ast.Module(body=helpers + data_nodes, type_ignores=[]), '<approved-functions-data>', 'exec'), globals())
print('skn25 | 검색 5:5 | E5 CPU | 부모 임베딩·LLM 호출 없음 | 기존 notebook 출력 실행 없음')


## 2. 공통 질문·정답 근거 목록 고정

복합 질문은 기존 근거 두 개를 함께 요구합니다. 예를 들어 화장품·생활용품·음료의 사업 구분과 계절적 매출 보완을 모두 확인해야 합니다.
정답 근거가 검색 질문이나 임베딩 입력에 들어가지 않으며, 평가할 때만 사용합니다. 새 회사·문서·독립 정답을 확보한 것은 아닙니다.

In [ ]:
reference_quotes = {case['id']: case['quote'] for case in CASES}
qcases = [{**{k: case[k] for k in ('id','company','question','topic')},
           'dataset_group': 'single_25', 'reference_ids': [case['id']]} for case in CASES]
qcases += [
    {'id':'M_S','company':'삼성전자','topic':'복합','dataset_group':'multi_5','reference_ids':['S04','S05'],
     'question':'삼성전자의 반도체 사업과 하만 전장 사업이 각각 어떤 제품을 만드는지 비교해줘.'},
    {'id':'M_R','company':'리노공업','topic':'복합','dataset_group':'multi_5','reference_ids':['R02','R05'],
     'question':'리노공업의 해외 거래처 관리 방식과 원자재·에너지 관련 위험을 함께 정리해줘.'},
    {'id':'M_N','company':'NAVER','topic':'복합','dataset_group':'multi_5','reference_ids':['N01','N02'],
     'question':'네이버 서치플랫폼과 쇼핑 사업의 수익원을 비교하고 쇼핑의 광고 외 수익도 설명해줘.'},
    {'id':'M_L','company':'LG생활건강','topic':'복합','dataset_group':'multi_5','reference_ids':['L01','L04'],
     'question':'LG생활건강의 사업 구분과 화장품·음료 사업의 계절적 매출 보완 관계를 함께 설명해줘.'},
    {'id':'M_H','company':'HDC','topic':'복합','dataset_group':'multi_5','reference_ids':['H01','H04'],
     'question':'HDC 지주회사 자체의 수익원과 아이파크몰 입점 사업자를 통한 임대수익 구조를 구분해줘.'},
]
assert len(qcases) == 30 and len({case['id'] for case in qcases}) == 30
assert all(ref in reference_quotes for case in qcases for ref in case['reference_ids'])
case_company = {case['id']: case['company'] for case in CASES}
assert all(case_company[ref] == case['company'] for case in qcases for ref in case['reference_ids'])
evaluation_sha256 = hashlib.sha256(json.dumps({'questions':qcases,'references':reference_quotes}, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
display(pd.DataFrame(qcases)[['id','company','dataset_group','question','reference_ids']])
print('고정 질문·근거 SHA256:', evaluation_sha256)


## 3. 동일한 공식 보고서 선택과 텍스트 추출

29번의 기업코드·제출일·사업연도·보고서명·접수번호·표지 검사 코드를 그대로 사용합니다.
다운로드는 이번 커널 메모리에서 처리하며 원본·인덱스의 운영 저장 정책을 새로 구현하지 않습니다. 키·요청 URL은 출력하지 않습니다.

In [ ]:
exec(compile(source_gate, '<29-approved-source-gate>', 'exec'), globals())
assert set(source_texts) == set(REPORTS)


## 4. 두 청킹 구성과 실제 부모·자식 연결

E5 기준으로 토큰 수를 세되, 부모는 E5 입력 한도보다 커도 검색 모델에 넣지 않습니다.
자식/작은 청크만 `passage:`를 포함해 512-token 이하인지 검사합니다. 두 후보 모두 같은 제목 섹션을 입력으로 사용합니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False, verbose=False))

def make_candidate(text, method):
    sections = [s for s in re.split(r'(?=^## )', text, flags=re.MULTILINE) if s.strip()]
    docs = [Document(text=s) for s in sections]
    small = SentenceSplitter(chunk_size=CHILD_SIZE, chunk_overlap=OVERLAP, tokenizer=lambda s: tokenizer.encode(s, add_special_tokens=False, verbose=False))
    if method == 'title_boundary':
        leaves = small.get_nodes_from_documents(docs)
        units = {n.node_id: n.text.strip() for n in leaves}
        payload_ids = [n.node_id for n in leaves]
    elif method == 'parent_child':
        parent = SentenceSplitter(chunk_size=PARENT_SIZE, chunk_overlap=OVERLAP, tokenizer=lambda s: tokenizer.encode(s, add_special_tokens=False, verbose=False))
        hierarchy = HierarchicalNodeParser.from_defaults(node_parser_ids=['parent','child'],
                    node_parser_map={'parent':parent,'child':small})
        nodes = hierarchy.get_nodes_from_documents(docs)
        leaves = get_leaf_nodes(nodes)
        parents = {n.node_id:n for n in nodes if n.child_nodes}
        units = {node_id:n.text.strip() for node_id,n in parents.items()}
        payload_ids = []
        for leaf in leaves:
            if not leaf.parent_node or leaf.parent_node.node_id not in parents:
                raise RuntimeError('자식에 연결된 부모를 찾지 못했습니다.')
            parent_id = leaf.parent_node.node_id
            if not any(n.node_id == leaf.node_id for n in parents[parent_id].child_nodes):
                raise RuntimeError('부모→자식 역방향 연결 오류')
            if compact(leaf.text.strip()) not in compact(units[parent_id]):
                raise RuntimeError('자식 텍스트가 연결된 부모 문맥에 없습니다.')
            payload_ids.append(parent_id)
    else:
        raise ValueError('알 수 없는 청킹 후보')
    texts = [n.text.strip() for n in leaves]
    if not texts or not all(texts):
        raise RuntimeError('빈 검색 청크가 있습니다.')
    lengths = [len(tokenizer('passage: ' + t, truncation=False, verbose=False)['input_ids']) for t in texts]
    if max(lengths) > 512:
        raise RuntimeError('E5에 넣을 작은 청크가 512-token을 넘습니다. 잘라내지 않고 중단합니다.')
    return texts, payload_ids, units, max(lengths)


## 5. 분할 결과와 근거 보존 검사

원문·A 청크·B 부모에 정답 인용문이 없으면 중단합니다. B 자식에서 인용문이 경계에 걸려 온전히 남지 않은 경우는 기록하되,
부모에 보존돼 있으면 제외하지 않습니다. 그런 경우야말로 부모 확장이 개선할 수 있는지를 볼 대상입니다.

In [ ]:
texts_by_key, payload_ids_by_key, context_units_by_key = {}, {}, {}
chunk_rows, audit_rows = [], []
for method in METHODS:
    for company, text in source_texts.items():
        key = (method, company)
        started = perf_counter()
        texts, payload_ids, units, max_tokens = make_candidate(text, method)
        texts_by_key[key], payload_ids_by_key[key], context_units_by_key[key] = texts, payload_ids, units
        chunk_rows.append({'method':method,'company':company,'search_chunks':len(texts),
                           'return_units':len(units),'max_e5_tokens':max_tokens,
                           'chunking_seconds':perf_counter()-started,
                           'search_text_MiB':sum(len(t.encode()) for t in texts)/1024**2,
                           'context_text_MiB':sum(len(t.encode()) for t in units.values())/1024**2})
        print(f'{method} | {company} | 검색 청크 {len(texts)} | 반환 단위 {len(units)}', flush=True)
        for case in [c for c in CASES if c['company'] == company]:
            needle = compact(case['quote'])
            audit_rows.append({'method':method,'id':case['id'],'company':company,
                'source_exists':needle in compact(text),
                'search_gold_chunks':sum(needle in compact(t) for t in texts),
                'return_gold_units':sum(needle in compact(t) for t in units.values())})
chunk_info, gold_audit = pd.DataFrame(chunk_rows), pd.DataFrame(audit_rows)
display(chunk_info); display(gold_audit)
bad = gold_audit[(~gold_audit['source_exists']) | (gold_audit['return_gold_units'] == 0)
                 | ((gold_audit['method'] == 'title_boundary') & (gold_audit['search_gold_chunks'] == 0))]
if not bad.empty:
    raise RuntimeError('원문·반환 문맥의 정답 근거 보존 실패. 문항을 빼지 말고 추출·분할을 확인하세요.')


## 6. 같은 Kiwi·BM25 검색 조건으로 두 인덱스 만들기

BM25는 A의 작은 청크/B의 자식 청크만 인덱싱합니다. 부모 텍스트를 BM25에 추가하지 않습니다.
회사와 후보별 메모리 DB를 분리해 다른 기업·다른 후보의 근거가 섞이지 않게 합니다.

In [ ]:
kiwi = Kiwi()
for connection in globals().get('bm25_connections', {}).values():
    connection.close()
bm25_connections, bm25_rows = {}, []
for key, texts in texts_by_key.items():
    started = perf_counter()
    connection = sqlite3.connect(':memory:')
    connection.execute('CREATE VIRTUAL TABLE passage USING fts5(text)')
    terms = [' '.join(morph_terms(text)) for text in texts]
    connection.executemany('INSERT INTO passage(rowid, text) VALUES (?,?)', enumerate(terms, 1))
    connection.commit()
    bm25_connections[key] = connection
    pages = connection.execute('PRAGMA page_count').fetchone()[0]
    page_size = connection.execute('PRAGMA page_size').fetchone()[0]
    bm25_rows.append({'method':key[0],'company':key[1],'bm25_build_seconds':perf_counter()-started,
                     'sqlite_MiB':pages*page_size/1024**2,'empty_bm25_chunks':sum(not t for t in terms)})
display(pd.DataFrame(bm25_rows))


## 7. E5 CPU 임베딩 — 부모는 임베딩하지 않음

E5를 한 번 로드합니다. 두 후보에 완전히 같은 청크 텍스트가 있으면 그 벡터만 재사용합니다.
문서 처리 진행률은 64개마다 표시합니다. 임베딩 준비 비용은 재사용 개수를 함께 해석해야 하며, 후보별 독립 cold 시간 비교가 아닙니다.

In [ ]:
started = perf_counter()
print('E5 CPU 모델 로드 시작', flush=True)
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_safetensors=True).cpu().eval()
model_load_seconds = perf_counter()-started
parameter_MiB = sum(p.numel()*p.element_size() for p in model.parameters())/1024**2
embedding_cache, dense_index, e5_rows = {}, {}, []
for key, texts in texts_by_key.items():
    started = perf_counter()
    missing = [t for t in dict.fromkeys(texts) if t not in embedding_cache]
    print(f'{key} | {len(texts)}청크 | 새 임베딩 {len(missing)}개', flush=True)
    for start in range(0, len(missing), 64):
        batch = missing[start:start+64]
        vectors = encode_texts(batch, 'passage')
        embedding_cache.update(zip(batch, vectors))
        print(f'  {min(start+64,len(missing))}/{len(missing)} 새 청크 완료', flush=True)
    dense_index[key] = np.stack([embedding_cache[t] for t in texts])
    e5_rows.append({'method':key[0],'company':key[1],'new_embedding_texts':len(missing),
                   'reused_records':len(texts)-len(missing),'e5_build_seconds':perf_counter()-started,
                   'vectors_MiB':dense_index[key].nbytes/1024**2})
display(pd.DataFrame(e5_rows))
print(f'E5 CPU 로드 {model_load_seconds:.2f}초 | 파라미터 메모리 {parameter_MiB:.1f} MiB')


## 8. 상위 5개에서 반환 문맥 만들기와 공통 예산

B는 검색된 자식의 부모를 순서대로 가져오고 동일 부모를 중복 제거합니다. A는 작은 청크 그대로입니다.
반환 문맥의 Recall은 정답 인용문 ID 목록 중 온전히 들어 있는 인용문 비율입니다. 이 점수는 LLM의 이해·답변 품질을 대신하지 않습니다.
예산은 E5 토크나이저의 텍스트 환산값이며 실제 운영 LLM의 입력 토큰이나 API 청구 토큰이 아닙니다.

In [ ]:
def returned_contexts(key, order, k=EVAL_K):
    unique, seen = [], set()
    for idx in order[:k]:
        unit_id = payload_ids_by_key[key][int(idx)]
        if unit_id not in seen:
            unique.append(context_units_by_key[key][unit_id])
            seen.add(unit_id)
    return unique

def pack_contexts(units, budget=CONTEXT_BUDGET):
    accepted = []
    for unit in units:
        if count_tokens('\n\n'.join(accepted + [unit])) > budget:
            break
        accepted.append(unit)
    return accepted

def evidence_recall(units, reference_ids):
    if not reference_ids:
        raise ValueError('정답 근거 목록이 비어 있습니다.')
    return sum(any(compact(reference_quotes[ref]) in compact(unit) for unit in units)
               for ref in reference_ids)/len(reference_ids)

# 실제 반환 정책 회귀 검사: 부모 중복 제거, 더 많은 자식으로 보충하지 않음, 예산 초과/부분 문맥 금지.
probe_key = ('probe','company')
payload_ids_by_key[probe_key] = ['p','p','q']
context_units_by_key[probe_key] = {'p':'AAAA','q':'BBBB'}
assert returned_contexts(probe_key, [0,1,2], k=2) == ['AAAA']
assert returned_contexts(probe_key, [0,1,2], k=3) == ['AAAA','BBBB']
assert pack_contexts(['AAAA','BBBB'], budget=count_tokens('AAAA')) == ['AAAA']
assert pack_contexts(['AAAA'], budget=0) == []
assert evidence_recall([reference_quotes['S04']], ['S04','S05']) == 0.5
assert evidence_recall([reference_quotes['S04'],reference_quotes['S05']], ['S04','S05']) == 1.0
del payload_ids_by_key[probe_key], context_units_by_key[probe_key]
print('부모 연결 반환·중복 제거·공통 예산·고정 근거 Recall 검사 완료')


## 9. 단일·복합 질문 A/B 실행

질문 임베딩은 두 후보가 공유합니다. 검색은 후보별 다른 청크 인덱스에서 수행합니다.
각 문항마다 **검색 청크 Hit / 부모 확장 뒤 근거 Recall / 공통 예산 뒤 Recall** 진행값을 출력합니다.
부모 확장 점수가 좋아도 제공 문맥이 훨씬 길어진 효과인지 확인해야 합니다.

In [ ]:
rows, details = [], []
for number, case in enumerate(qcases, 1):
    print(f"[{number}/{len(qcases)}] {case['id']} {case['company']} 검색 시작", flush=True)
    started = perf_counter()
    query_vector = encode_texts([case['question']], 'query')[0]
    query_embed_ms = (perf_counter()-started)*1000
    for method in METHODS:
        key, refs = (method,case['company']), case['reference_ids']
        started = perf_counter()
        lex_order = bm25_order(key, case['question'])
        dense_order = np.argsort(-(dense_index[key] @ query_vector), kind='stable')
        order = weighted_rrf(dense_order, lex_order, len(texts_by_key[key]), VECTOR_WEIGHT)
        search_ms = query_embed_ms + (perf_counter()-started)*1000
        first_rank = next((rank for rank, idx in enumerate(order, 1)
                          if evidence_recall([texts_by_key[key][int(idx)]], refs) > 0), None)
        leaf_units = [texts_by_key[key][int(idx)] for idx in order[:EVAL_K]]
        started = perf_counter()
        contexts = returned_contexts(key, order)
        bounded = pack_contexts(contexts)
        context_ms = (perf_counter()-started)*1000
        raw_recall, bounded_recall = evidence_recall(contexts, refs), evidence_recall(bounded, refs)
        rows.append({'id':case['id'],'company':case['company'],'dataset_group':case['dataset_group'],
                     'method':method,'references':len(refs),'first_leaf_rank':first_rank,
                     'leaf_hit_at_5':first_rank is not None and first_rank <= EVAL_K,
                     'leaf_mrr_at_5':1/first_rank if first_rank is not None and first_rank <= EVAL_K else 0.0,
                     'leaf_evidence_recall_at_5':evidence_recall(leaf_units, refs),
                     'context_hit_at_5':raw_recall > 0,'context_recall_at_5':raw_recall,
                     'budget_context_hit_at_5':bounded_recall > 0,'budget_context_recall_at_5':bounded_recall,
                     'raw_contexts':len(contexts),'packed_contexts':len(bounded),
                     'raw_e5_tokens':count_tokens('\n\n'.join(contexts)),
                     'packed_e5_tokens':count_tokens('\n\n'.join(bounded)),
                     'search_ms':search_ms,'context_ms':context_ms})
        missing = [ref for ref in refs if evidence_recall(bounded,[ref]) == 0]
        details.append({'id':case['id'],'company':case['company'],'method':method,
                        'question':case['question'],'missing_reference_ids':missing,
                        'reference_quotes':[reference_quotes[r] for r in refs],
                        'top1_leaf':leaf_units[0][:240] if leaf_units else '',
                        'first_context_preview':contexts[0][:320] if contexts else ''})
        print(f"  {method}: leaf Hit={rows[-1]['leaf_hit_at_5']} | 문맥 Recall={raw_recall:.2f} | 예산 Recall={bounded_recall:.2f} | {rows[-1]['packed_e5_tokens']}tokens", flush=True)
results, detail_df = pd.DataFrame(rows), pd.DataFrame(details)
assert len(results) == 60 and results.groupby('method').size().eq(30).all()
assert results['packed_e5_tokens'].le(CONTEXT_BUDGET).all()
display(results)


## 10. 전체·질문 그룹·기업별 비교와 실패 확인

주요 판단은 **공통 예산 안에서 확보한 고정 근거 Recall**과 Hit입니다. 첫 근거 순위는 보조로 봅니다.
부모가 길다는 사실만으로 품질 우위를 주장하지 않습니다. 복합 질문은 5개뿐이라 방향 확인용이며 일반화할 수 없습니다.

In [ ]:
summary = results.groupby('method').agg(
    questions=('id','count'), leaf_hit_at_5=('leaf_hit_at_5','mean'), leaf_mrr_at_5=('leaf_mrr_at_5','mean'),
    leaf_evidence_recall_at_5=('leaf_evidence_recall_at_5','mean'),
    context_hit_at_5=('context_hit_at_5','mean'), context_recall_at_5=('context_recall_at_5','mean'),
    budget_context_hit_at_5=('budget_context_hit_at_5','mean'), budget_context_recall_at_5=('budget_context_recall_at_5','mean'),
    median_raw_tokens=('raw_e5_tokens','median'), median_packed_tokens=('packed_e5_tokens','median'),
    median_search_ms=('search_ms','median'), p95_search_ms=('search_ms',lambda s:s.quantile(.95)),
    median_context_ms=('context_ms','median')).reindex(METHODS)
display(summary.T)
print('단일 25개와 복합 5개 결과 분리')
display(results.groupby(['dataset_group','method'])[['leaf_hit_at_5','leaf_evidence_recall_at_5',
    'budget_context_hit_at_5','budget_context_recall_at_5','raw_e5_tokens','packed_e5_tokens']].mean())
print('기업별 결과 — 기업당 6개 질문')
display(results.groupby(['company','method'])[['budget_context_hit_at_5','budget_context_recall_at_5']].mean())
print('동일 질문별 공통 예산 근거 Recall')
display(results.pivot(index=['company','id'], columns='method', values='budget_context_recall_at_5'))
print('부모 확장으로 회복했거나 예산 적용으로 근거가 빠진 사례')
changed = results[(results['context_recall_at_5'] != results['leaf_evidence_recall_at_5'])
                  | (results['budget_context_recall_at_5'] != results['context_recall_at_5'])]
display(changed[['id','method','leaf_evidence_recall_at_5','context_recall_at_5','budget_context_recall_at_5','raw_e5_tokens','packed_e5_tokens']])
failed = results.loc[results['budget_context_recall_at_5'] < 1, ['id','method']]
with pd.option_context('display.max_colwidth', None, 'display.max_rows', 60):
    display(failed.merge(detail_df, on=['id','method']))


## 11. 재현·자원 기록과 해석 기준

임베딩 파라미터 메모리·벡터 배열·텍스트 크기·SQLite 페이지 크기는 전체 RAM peak가 아닙니다.
검색 시간은 인덱스 준비 뒤의 로컬 구성 시간이며 최종 사용자 응답 시간은 아닙니다. 부모 확장의 LLM 비용은 호출하지 않아 측정하지 않습니다.
기록된 E5-token은 향후 운영 모델의 토크나이저로 다시 확인해야 합니다.

In [ ]:
runtime = {'observed_at_utc':datetime.now(timezone.utc).isoformat(), 'dataset_type':'development',
    'companies':len(REPORTS),'reports':len(REPORTS),'questions':len(qcases), 'single_questions':25,'multi_questions':5,
    'evaluation_sha256':evaluation_sha256,'reused_ast_sha256':reuse_sha256,
    'embedding_model':MODEL_ID,'model_revision':MODEL_REVISION,'device':'cpu',
    'vector_bm25_weights':[0.5,0.5],'candidate_depth_each':CANDIDATE_DEPTH,'rrf_k':RRF_K,
    'child_tokens':CHILD_SIZE,'parent_tokens':PARENT_SIZE,'overlap':OVERLAP,'retrieved_leaves':EVAL_K,
    'context_budget_e5_tokens':CONTEXT_BUDGET,'context_packing':'whole units, ranked prefix, stop before overflow',
    'model_parameter_MiB':parameter_MiB,'model_load_seconds':model_load_seconds,
    'llm_calls':0,'openai_embedding_calls':0,
    'packages':{p:importlib.metadata.version(p) for p in ['llama-index-core','transformers','torch','kiwipiepy']}}
print(json.dumps(runtime,ensure_ascii=False,indent=2))
display(chunk_info)
display(pd.DataFrame(bm25_rows)); display(pd.DataFrame(e5_rows))
print('외부 LLM/임베딩 API 과금 없음. 로컬 CPU·RAM·전력, OpenDART 사용 제한은 별도입니다.')
print('최종 청킹·운영 구조는 자동 채택하지 않습니다. 저장 후 결과를 함께 해석합니다.')


## 결과를 읽을 때

- B의 leaf 검색이 좋아졌다면 계층 분할로 검색 청크 경계가 바뀐 효과를 포함합니다.
- leaf는 비슷한데 반환 문맥 Recall이 좋아졌다면 부모 확장이 근거를 함께 가져온 효과입니다.
- 무제한 문맥에서만 좋아지고 같은 예산에서 좋아지지 않으면 길이 증가와 품질 향상을 구분해야 합니다.
- 지표가 같아도 주변 설명·표 제목을 실제로 더 잘 활용하는지는 **후속 LLM 답변 평가**가 필요합니다.
- 이번 30개 개발 질문만으로 청킹 최적 크기·최종 검색·답변 품질·할루시네이션을 확정하지 않습니다.

### 근거 자료 — 2026-10-05 확인

- [LlamaIndex HierarchicalNodeParser 공식 API](https://developers.llamaindex.ai/python/framework-api-reference/node_parsers/hierarchical/): 큰 부모와 작은 자식 Node 관계를 생성합니다.
- [LlamaIndex Auto Merging 공식 예제](https://developers.llamaindex.ai/python/framework/integrations/retrievers/auto_merging_retriever/): leaf를 인덱싱하고 부모 문맥을 별도로 보존하는 구조의 근거입니다. 임계값 병합 클래스 자체는 이번에 사용하지 않습니다.
- [29번 notebook](29_rrf_weight_comparison.ipynb): 데이터·원문 선택·검색 가중치 기준선.
- [평가 지표 정의](../../docs/EVALUATION_METRICS.md): 반환 문맥 단계의 Hit·근거 Recall과 토큰 예산을 구분합니다.